# Add group type tasks to LabelStudio

In [1]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from label_studio_sdk.client import LabelStudio
import sys

# Load environment variables
load_dotenv()

# Direct the notebook to find scripts in parent directory, since it is sitting one level down
os.chdir("..")

sys.path.append(os.getcwd())

# Import the new task classes
from adt_labelstudio.group_type import GroupTypeTask
from adt_labelstudio.utils import get_project_annotations, get_ls_project_id_from_name

# Utility functions for normalization (if not already available from utils)
from adt_eval.utils.transcript_cleaner import standardize_transcript, normalize_transcript


In [2]:
# Connect to the Label Studio API and check the connection
LABEL_STUDIO_URL = "https://" + os.getenv("LABEL_STUDIO_HOST")
API_KEY = os.getenv("LABEL_STUDIO_TOKEN")

ls_client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=API_KEY)

group_task = GroupTypeTask()

In [3]:
file_dir = "output/evals/logs/text_extraction/"
file_list = os.listdir(file_dir)

source_project_name = "A1: Text Extraction"
target_project_name = "A3: Group type"

# Get annotations from previous task, used to populate this one
gs_annotations = get_project_annotations(ls_client, project_name=source_project_name) 

# Get annotations that have already been done for this task
target_project_tasks = get_project_annotations(ls_client, project_name=target_project_name)

tasks_to_load = []

for f in file_list:
    llm_log, book_id, page_id = group_task.get_llm_log(f"{file_dir}/{f}")
    print(f)
    
    # Confirm that task does not already exist
    if target_project_tasks.shape[0]>0 and (book_id, page_id) in [xy for xy in zip(target_project_tasks['book_id'], target_project_tasks['page_id'])]:
        print(f"Task for {book_id} page {page_id} already exists in LabelStudio and was not added.")
        continue

    print(f"Processing task for {book_id} page {page_id}.")

    # Read in the LLM log file to dataframe
    llm_df = group_task.load_llm_log_to_df(llm_log)

    # Get the Gold Standard data as dataframe
    gs_annotation = group_task.get_single_annotation(gs_annotations, book_id, page_id)
    gs_df = group_task.load_gs_annotation_to_df(gs_annotation)
    
    # Merge Gold Standard and LLM dataframes on exact match
    matched_df = group_task.merge_gs_with_llm(gs_df, llm_df)
    
    # Create labelstudio task, consisting of input data and predictions
    task_data = group_task.populate_task_data(gs_annotation)
    task_predictions = group_task.populate_task_predictions(matched_df)
    task_json = group_task.create_one_task(task_data, task_predictions)

    tasks_to_load.append(task_json)

# Add the whole list to LabelStudio
if len(tasks_to_load) > 0:
    target_project_id  = get_ls_project_id_from_name(ls_client, project_name=target_project_name)
    ls_client.projects.import_tasks(
                id=target_project_id,
                request=tasks_to_load,
            )

with open("adt_labelstudio/tasks_to_upload/group_type_tasks.json", "w") as f:                  
    json.dump(tasks_to_load, f)

text_extraction_eval_29320.json
Processing task for B-54 page 1.
text_extraction_eval_29321.json
Processing task for B-54 page 3.
text_extraction_eval_29322.json
Processing task for B-89 page 3.
text_extraction_eval_29323.json
Processing task for B-89 page 7.
text_extraction_eval_29324.json
Processing task for B-89 page 9.
text_extraction_eval_29325.json
Processing task for B-89 page 14.
Groupings are invalid: groups are shuffled together. No pre-annotations added.
text_extraction_eval_29326.json
Processing task for B-89 page 16.
Groupings are invalid: groups are shuffled together. No pre-annotations added.
text_extraction_eval_29327.json
Processing task for B-89 page 17.
text_extraction_eval_29328.json
Processing task for B-89 page 23.
text_extraction_eval_29329.json
Processing task for B-89 page 25.
text_extraction_eval_29330.json
Processing task for B-89 page 30.
Groupings are invalid: groups are shuffled together. No pre-annotations added.
text_extraction_eval_29331.json
Processing

In [4]:
task_json

{'data': {'book_id': 'W-38',
  'page_id': 0,
  'page_image': 'azure-blob://adt-pipeline/evaluation/gold_standard/pages/W-38__page_1.png',
  'page_text_all': 'U.S. EDITION\n\nPRIMARY MATHEMATICS 1A\n\nTEXTBOOK\n\nMarshall Cavendish Education\n\nSingaporeMath.com Inc'},
 'predictions': [{'result': [{'value': {'text': 'textbook',
      'taxonomy': [['heading']],
      'start': 38,
      'end': 46},
     'from_name': 'group_type_annotations',
     'to_name': 'page_text_all',
     'type': 'taxonomy'},
    {'value': {'text': 'us edition',
      'taxonomy': [['other']],
      'start': 0,
      'end': 12},
     'from_name': 'group_type_annotations',
     'to_name': 'page_text_all',
     'type': 'taxonomy'},
    {'value': {'text': 'marshall cavendish education',
      'taxonomy': [['other']],
      'start': 48,
      'end': 76},
     'from_name': 'group_type_annotations',
     'to_name': 'page_text_all',
     'type': 'taxonomy'},
    {'value': {'text': 'singaporemathcom inc',
      'taxonomy': 